# Парсинг данных ТОП 250-Фильмов

### Подключение библиотек

In [153]:
from bs4 import BeautifulSoup as bs
import requests
import pandas as pd
import time
import re

### URL ссылка сайта и получение информации

# Загрузка главной страницы

In [154]:
url = 'https://www.kinoafisha.info/rating/movies/imdb/'

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

response = requests.get(url, headers=headers, timeout=30)
print(f"Статус: {response.status_code}")

soup = bs(response.text, 'html.parser')

Статус: 200


### Парсинг

In [155]:
movies = soup.find_all('div', class_='movieList_item')
print(f"Всего блоков: {len(movies)}")

result_list = {
    'title': [], 'rating': [], 'year': [], 'country': [], 'description': []
}

for i, movie in enumerate(movies[:250]):
    # Рейтинг
    rating_elem = movie.find('span', class_='movieItem_itemRating')
    rating = rating_elem.get_text(strip=True) if rating_elem else ''
    
    title_elem = movie.find(class_='movieItem_title')
    if not title_elem:
        title = ''
        film_url = ''
    else:
        title = title_elem.get_text(strip=True)
        film_url = title_elem.get('href', '') if title_elem.name == 'a' else ''
    
    # Год и страна
    year_span = movie.find('span', class_='movieItem_year')
    if year_span:
        year_text = year_span.get_text(strip=True)
        parts = year_text.split(',')
        year = parts[0].strip()
        country = parts[1].strip() if len(parts) > 1 else ''
    else:
        year = ''
        country = ''
    
    # Описание
    description = ''
    if film_url:
        if not film_url.startswith('http'):
            film_url = 'https://www.kinoafisha.info' + film_url
        try:
            film_resp = requests.get(film_url, headers=headers, timeout=10)
            if film_resp.status_code == 200:
                film_soup = bs(film_resp.text, 'html.parser')
                desc_div = film_soup.find('div', class_='visualEditorInsertion-info')
                if desc_div:
                    description = desc_div.get_text(strip=True)
                    description = re.sub(r'\s+', ' ', description)
        except:
            pass
    
    result_list['title'].append(title)
    result_list['rating'].append(rating)
    result_list['year'].append(year)
    result_list['country'].append(country)
    result_list['description'].append(description)
    
    print(f"{i+1}. {title} ({year}) - {country}")

print(f"\nСохранено {len(result_list['title'])} фильмов")

df = pd.DataFrame(result_list)

Всего блоков: 250
1. Побег из Шоушенка (1994) - США
2. Крестный отец (1972) - США
3. Темный рыцарь (2008) - США
4. Крестный отец 2 (1974) - США
5. 12 разгневанных мужчин (1957) - США
6. Властелин Колец: Возвращение Короля (2003) - США / Новая Зеландия / Германия
7. Список Шиндлера (1993) - США
8. Властелин Колец: Братство Кольца (2001) - Новая Зеландия / США
9. Криминальное чтиво (1994) - США
10. Хороший, плохой, злой (1966) - Италия / Испания / Германия
11. Властелин Колец: Две крепости (2002) - США / Новая Зеландия / Германия
12. Форрест Гамп (1994) - США
13. Бойцовский клуб (1999) - США / Германия
14. Начало (2010) - США / Великобритания
15. Звездные войны: Эпизод V - Империя наносит ответный удар (1980) - США
16. Матрица (1999) - Австралия / США
17. Славные парни (1990) - США
18. Интерстеллар (2014) - США / Великобритания
19. Пролетая над гнездом кукушки (1975) - США
20. Семь (1995) - США
21. Эта замечательная жизнь (1946) - США
22. Молчание ягнят (1991) - США
23. Семь самураев (19

In [156]:
df = pd.DataFrame(result_list)

### Вывод таблицы

In [157]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   title        250 non-null    object
 1   rating       250 non-null    object
 2   year         250 non-null    object
 3   country      250 non-null    object
 4   description  250 non-null    object
dtypes: object(5)
memory usage: 9.9+ KB


In [158]:
df.head(10)

,title,rating,year,country,description
0,Побег из Шоушенка,9.3,1994,США,Успешный банковский служащий Энди Дюфрейн осуж...
1,Крестный отец,9.2,1972,США,В центре культовой криминальной драмы Фрэнсиса...
2,Темный рыцарь,9.1,2008,США,Действие фильма начинается с ограбления Готэмс...
3,Крестный отец 2,9.0,1974,США,Молодой Вито теряет семью из-за преследования ...
4,12 разгневанных мужчин,9.0,1957,США,"Классика кинематографа 1950-х, выросшего из те..."
5,Властелин Колец: Возвращение Короля,9.0,2003,США / Новая Зеландия / Германия,Битва за Средиземье продолжается. Хоббиты и их...
6,Список Шиндлера,9.0,1993,США,"Начало Второй мировой войны. После того, как П..."
7,Властелин Колец: Братство Кольца,8.9,2001,Новая Зеландия / США,"Средиземье — континент, населенный разными и о..."
8,Криминальное чтиво,8.8,1994,США,Джулс Уиннфилд и Винсент Вега – парочка киллер...
9,"Хороший, плохой, злой",8.8,1966,Италия / Испания / Германия,В разгар гражданской войны таинственный стрело...


### Сохранение в csv файл

In [159]:
df.to_csv('top250.csv', index=False, encoding='utf-8-sig')

In [3]:
import pandas as pd

In [4]:
df = pd.read_csv('top250.csv')

In [6]:
df['country'].isna().sum()

np.int64(5)

In [7]:
df['country'].isnull().sum()

np.int64(5)

In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   title        500 non-null    object 
 1   rating       500 non-null    float64
 2   year         500 non-null    int64  
 3   country      495 non-null    object 
 4   description  494 non-null    object 
dtypes: float64(1), int64(1), object(3)
memory usage: 19.7+ KB


### Парсинг с помощью API к кинопоиску

In [160]:
API_TOKEN = "QKWZN69-DNS4ZWV-HDG2822-RWXZH9Q"
headers = {'X-API-KEY': API_TOKEN, 'Content-Type': 'application/json'}
result_list = {'title': [], 'year': [], 'country': [], 'rating': [], 'description': []}

In [161]:
page = 1
collected = 0
max_attempts = 10
attempts = 0

while collected < 250 and attempts < max_attempts:
    try:
        url = f"https://api.kinopoisk.dev/v1.4/movie?lists=top250&limit=50&page={page}"
        
        response = requests.get(url, headers=headers, timeout=10)
        
        if response.status_code != 200:
            print(f"Ошибка API: статус {response.status_code}")
            attempts += 1
            time.sleep(2)
            continue
            
        data = response.json()
        movies = data.get('docs', [])
        
        for movie in movies:
            if collected >= 250:
                break
            
            # Название
            title = movie.get('name') or movie.get('alternativeName', 'Не указано')
            
            # Год
            year = movie.get('year', 'Не указан')
            
            # Страна
            countries = movie.get('countries', [])
            country = countries[0].get('name', 'Не указана') if countries else 'Не указана'
            
            # Рейтинг
            rating_data = movie.get('rating', {})
            rating = rating_data.get('kp', None)
            if rating:
                rating = str(round(float(rating), 1))
            else:
                rating = 'Нет рейтинга'
            
            # Описание
            description = movie.get('description') or movie.get('shortDescription', 'Описание отсутствует')
            
            result_list['title'].append(title)
            result_list['year'].append(year)
            result_list['country'].append(country)
            result_list['rating'].append(rating)
            result_list['description'].append(description)
            
            collected += 1
            print(f"{collected}. {title[:30]} ({year}) - {country}")
        
        page += 1
        attempts = 0
        time.sleep(0.5)
        
    except requests.exceptions.Timeout:
        print(f"Таймаут на странице {page}, пробуем снова...")
        attempts += 1
        time.sleep(3)
    except requests.exceptions.RequestException as e:
        print(f"Сетевая ошибка: {e}")
        attempts += 1
        time.sleep(3)
    except Exception as e:
        print(f"Неожиданная ошибка: {e}")
        attempts += 1
        time.sleep(2)

print(f"\nСобрано {collected} фильмов из 250")

# Создание DataFrame
df_new = pd.DataFrame(result_list)

1. Батя (2020) - Россия
2. Крик тишины (2019) - Россия
3. Темные воды (2019) - США
4. Уроки фарси (2020) - Россия
5. Подольские курсанты (2020) - Россия
6. Достать ножи (2019) - США
7. Огонь (2020) - Россия
8. Дело Коллини (2019) - Германия
9. Джентльмены (2019) - США
10. Пальма (2020) - Россия
11. Звук свободы (2023) - Мексика
12. Зеленая книга (2018) - США
13. Тайна Коко (2017) - США
14. Гонка (2013) - Великобритания
15. Брестская крепость (2010) - Беларусь
16. Диодорова. Против течения (2024) - Россия
17. Истребитель демонов: Бесконечн (2025) - Япония
18. Боб Тревино поставил лайк (2024) - США
19. Нэчжа побеждает Царя драконов (2025) - Китай
20. Сводишь с ума (2025) - Россия
21. Дикий робот (2024) - США
22. Граф Монте-Кристо (2024) - Франция
23. Нюрнберг. Многомерность зла (2023) - Россия
24. Группа крови (2025) - Россия
25. Как приручить дракона (2025) - США
26. Одна жизнь (2023) - Великобритания
27. Роднина (2025) - Россия
28. Земмельвейс (2023) - Венгрия
29. F1 (2025) - США
30. О

In [162]:
df_new.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 250 entries, 0 to 249
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   title        250 non-null    object
 1   year         250 non-null    int64 
 2   country      250 non-null    object
 3   rating       250 non-null    object
 4   description  250 non-null    object
dtypes: int64(1), object(4)
memory usage: 9.9+ KB


In [163]:
df_new.head(10)

,title,year,country,rating,description
0,Батя,2020,Россия,7.9,История о путешествии взрослого героя к своему...
1,Крик тишины,2019,Россия,8.6,"Блокадный Ленинград, февраль 1942 года. Заканч..."
2,Темные воды,2019,США,7.9,Адвокат Роберт Билотт выступает с обвинением в...
3,Уроки фарси,2020,Россия,8.0,"1942 год, оккупированная Европа. Оказавшись в ..."
4,Подольские курсанты,2020,Россия,8.4,"Октябрь 1941 года, Подмосковье. Около трёх с п..."
5,Достать ножи,2019,США,8.2,На следующее утро после празднования 85-летия ...
6,Огонь,2020,Россия,8.1,После гибели подчинённого бывалый инструктор б...
7,Дело Коллини,2019,Германия,8.0,"Берлин, 2001 год. Молодой адвокат Каспар Лайне..."
8,Джентльмены,2019,США,8.7,Один ушлый американец ещё со студенческих лет ...
9,Пальма,2020,Россия,8.4,Овчарка по кличке Пальма вынужденно расстается...


### Объединение фильмов в 1 файл

In [164]:
file_name = 'top250.csv'

try:
    df_existing = pd.read_csv(file_name, encoding='utf-8-sig')
    df_combined = pd.concat([df_existing, df_new], ignore_index=True)
    df_combined.to_csv(file_name, index=False, encoding='utf-8-sig')
    print(f"Данные добавлены. Было: {len(df_existing)}, стало: {len(df_combined)}")
except FileNotFoundError:
    df_new.to_csv(file_name, index=False, encoding='utf-8-sig')
    print(f"Создан новый файл '{file_name}' с {len(df_new)} фильмами")

Данные добавлены. Было: 250, стало: 500


### Удаление дубликатов

In [165]:
file_name = 'top250.csv'
new_file_name = 'top250_no_duplicates.csv'

df = pd.read_csv(file_name, encoding='utf-8-sig')
print(f"Было: {len(df)}")

duplicates = df[df.duplicated(subset=['title', 'year', 'country'], keep=False)]

if len(duplicates) > 0:
    print(f"Найдено дубликатов: {len(duplicates)}")
    print("\nВсе дублирующиеся записи:")
    print(duplicates[['title', 'year', 'country', 'rating']].to_string(index=False))
    
    # Удаляем дубликаты
    df_cleaned = df.drop_duplicates(subset=['title', 'year', 'country'])
    print(f"\nСтало: {len(df_cleaned)}")
    
    df_cleaned.to_csv(new_file_name, index=False, encoding='utf-8-sig')
    print(f"Создан файл: {new_file_name}")
else:
    print("Дубликатов не найдено")

Было: 500
Найдено дубликатов: 100

Все дублирующиеся записи:
                        title  year        country  rating
                Крестный отец  1972            США     9.2
                Темный рыцарь  2008            США     9.1
              Список Шиндлера  1993            США     9.0
           Криминальное чтиво  1994            США     8.8
                 Форрест Гамп  1994            США     8.8
                Славные парни  1990            США     8.7
 Пролетая над гнездом кукушки  1975            США     8.6
                         Семь  1995            США     8.6
       Спасти рядового Райана  1998            США     8.6
                 Зеленая миля  1999            США     8.6
              Назад в будущее  1985            США     8.5
                   Отступники  2006            США     8.5
                         Леон  1994        Франция     8.5
                          1+1  2011        Франция     8.5
         Джанго освобожденный  2012            США    

### Просмотр общей информации о сохраненных данных

In [166]:
df_cleaned = pd.read_csv(new_file_name, encoding='utf-8-sig')
df_cleaned.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 450 entries, 0 to 449
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   title        450 non-null    object 
 1   rating       450 non-null    float64
 2   year         450 non-null    int64  
 3   country      445 non-null    object 
 4   description  444 non-null    object 
dtypes: float64(1), int64(1), object(3)
memory usage: 17.7+ KB


In [167]:
df.head(10)

,title,rating,year,country,description
0,Побег из Шоушенка,9.3,1994,США,Успешный банковский служащий Энди Дюфрейн осуж...
1,Крестный отец,9.2,1972,США,В центре культовой криминальной драмы Фрэнсиса...
2,Темный рыцарь,9.1,2008,США,Действие фильма начинается с ограбления Готэмс...
3,Крестный отец 2,9.0,1974,США,Молодой Вито теряет семью из-за преследования ...
4,12 разгневанных мужчин,9.0,1957,США,"Классика кинематографа 1950-х, выросшего из те..."
5,Властелин Колец: Возвращение Короля,9.0,2003,США / Новая Зеландия / Германия,Битва за Средиземье продолжается. Хоббиты и их...
6,Список Шиндлера,9.0,1993,США,"Начало Второй мировой войны. После того, как П..."
7,Властелин Колец: Братство Кольца,8.9,2001,Новая Зеландия / США,"Средиземье — континент, населенный разными и о..."
8,Криминальное чтиво,8.8,1994,США,Джулс Уиннфилд и Винсент Вега – парочка киллер...
9,"Хороший, плохой, злой",8.8,1966,Италия / Испания / Германия,В разгар гражданской войны таинственный стрело...
